# Hit vs Miss Neural State-Space Analysis From Google Drive


In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_package("pyarrow")
ensure_package("gdown")

from pathlib import Path
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/1NGZE-uG7GTJnLeR2P6ip98249lJeYC8d/view?usp=share_link"
PARQUET_PATH = Path("data/allen_visp_sst_vip_slc17a7_pilot.parquet")

PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
if not PARQUET_PATH.exists():
    downloaded_path = gdown.download(url=GDRIVE_URL, output=str(PARQUET_PATH), quiet=False, fuzzy=True)
    if downloaded_path is None:
        raise RuntimeError("Download failed. Check that the Drive file is shared with anyone who has the link.")

print(f"Using parquet: {PARQUET_PATH.resolve()}")

In [ ]:
data = pd.read_parquet(PARQUET_PATH)
required = {"stimulus_presentations_id", "cell_specimen_id", "trace", "trace_timestamps", "is_change", "behavior_outcome", "ophys_experiment_id"}
missing = sorted(required - set(data.columns))
if missing:
    raise ValueError(f"Parquet is missing Allen analysis column(s): {missing}")

change_data = data[data["is_change"].fillna(False).astype(bool)].copy()
change_data = change_data[change_data["behavior_outcome"].isin(["hit", "miss"])]
trial_counts = (change_data.drop_duplicates(["ophys_experiment_id", "stimulus_presentations_id"])
                .groupby(["ophys_experiment_id", "behavior_outcome"]).size().unstack(fill_value=0)
                .reindex(columns=["hit", "miss"], fill_value=0))
eligible = trial_counts[(trial_counts["hit"] > 0) & (trial_counts["miss"] > 0)]
if eligible.empty:
    raise ValueError("No experiment in this parquet contains both hit and miss change trials.")

experiment_id = eligible.sum(axis=1).idxmax()
experiment_data = change_data[change_data["ophys_experiment_id"] == experiment_id].copy()
display(eligible.loc[[experiment_id]])
print(f"Selected experiment: {experiment_id}")

In [ ]:
trial_meta = (experiment_data[["stimulus_presentations_id", "behavior_outcome"]]
              .drop_duplicates("stimulus_presentations_id").sort_values("stimulus_presentations_id"))
trial_ids = trial_meta["stimulus_presentations_id"].tolist()

cell_sets = [set(group["cell_specimen_id"].dropna()) for _, group in experiment_data.groupby("stimulus_presentations_id")]
common_cells = sorted(set.intersection(*cell_sets))
if not common_cells:
    raise ValueError("The selected trials have no common cells.")

activity = []
for trial_id in trial_ids:
    rows = (experiment_data[experiment_data["stimulus_presentations_id"] == trial_id]
            .set_index("cell_specimen_id").loc[common_cells])
    activity.append(np.stack([np.asarray(trace, dtype=float) for trace in rows["trace"]]))

all_activity = np.stack(activity)  # trials x neurons x time
rel_time = np.asarray(experiment_data.iloc[0]["trace_timestamps"], dtype=float)
if all_activity.shape[2] != len(rel_time):
    raise ValueError("Trace and timestamp lengths do not match.")

outcomes = trial_meta["behavior_outcome"].to_numpy()
print("all_activity (trials, neurons, time):", all_activity.shape)
print("hit trials:", int((outcomes == "hit").sum()))
print("miss trials:", int((outcomes == "miss").sum()))

In [ ]:
n_trials, n_neurons, n_time = all_activity.shape
X = all_activity.transpose(0, 2, 1).reshape(-1, n_neurons)
column_means = np.nan_to_num(np.nanmean(X, axis=0), nan=0.0)
nan_rows, nan_cols = np.where(np.isnan(X))
X[nan_rows, nan_cols] = column_means[nan_cols]

X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True)
X_std[X_std == 0] = 1
X_z = (X - X_mean) / X_std

U, S, Vt = np.linalg.svd(X_z, full_matrices=False)
scores = U * S
explained_variance_ratio = S**2 / np.sum(S**2)
print("Explained variance ratio, first 5 PCs:", explained_variance_ratio[:5])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
n_show = min(20, len(explained_variance_ratio))
ax.plot(np.arange(1, n_show + 1), explained_variance_ratio[:n_show], marker="o")
ax.set(xlabel="Principal component", ylabel="Explained variance ratio", title="PCA scree plot")
plt.show()

In [ ]:
scores_reshaped = scores[:, :3].reshape(n_trials, n_time, 3)
hit_scores = scores_reshaped[outcomes == "hit"]
miss_scores = scores_reshaped[outcomes == "miss"]
hit_traj = hit_scores.mean(axis=0)
miss_traj = miss_scores.mean(axis=0)
event_idx = np.argmin(np.abs(rel_time))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(hit_traj[:, 0], hit_traj[:, 1], label="Hit")
ax.plot(miss_traj[:, 0], miss_traj[:, 1], label="Miss")
ax.scatter(hit_traj[event_idx, 0], hit_traj[event_idx, 1], s=90, label="Hit change time")
ax.scatter(miss_traj[event_idx, 0], miss_traj[event_idx, 1], s=90, label="Miss change time")
ax.set(xlabel="PC1", ylabel="PC2", title="State-space trajectory: hit vs miss")
ax.axis("equal")
ax.legend()
plt.show()

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(hit_traj[:, 0], hit_traj[:, 1], hit_traj[:, 2], label="Hit")
ax.plot(miss_traj[:, 0], miss_traj[:, 1], miss_traj[:, 2], label="Miss")
ax.set(xlabel="PC1", ylabel="PC2", zlabel="PC3", title="3D state-space trajectory")
ax.legend()
plt.show()

In [ ]:
dt = np.median(np.diff(rel_time))

def compute_speed_and_acceleration(trajectory, sample_interval):
    velocity = np.diff(trajectory, axis=0) / sample_interval
    acceleration_vector = np.diff(velocity, axis=0) / sample_interval
    return np.linalg.norm(velocity, axis=1), np.linalg.norm(acceleration_vector, axis=1)

hit_speed, hit_acceleration = compute_speed_and_acceleration(hit_traj, dt)
miss_speed, miss_acceleration = compute_speed_and_acceleration(miss_traj, dt)
speed_time = rel_time[:-1]
acceleration_time = rel_time[:-2]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
axes[0].plot(speed_time, hit_speed, label="Hit")
axes[0].plot(speed_time, miss_speed, label="Miss")
axes[0].set(ylabel="Speed", title="Trajectory speed")
axes[1].plot(acceleration_time, hit_acceleration, label="Hit")
axes[1].plot(acceleration_time, miss_acceleration, label="Miss")
axes[1].set(xlabel="Time from change onset (s)", ylabel="Acceleration", title="Trajectory acceleration")
for ax in axes:
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
post_speed = (speed_time >= 0) & (speed_time <= 1.0)
post_acceleration = (acceleration_time >= 0) & (acceleration_time <= 1.0)
summary = pd.DataFrame({
    "hit": [hit_speed[post_speed].mean(), hit_acceleration[post_acceleration].mean()],
    "miss": [miss_speed[post_speed].mean(), miss_acceleration[post_acceleration].mean()],
}, index=["post-change mean speed", "post-change mean acceleration"])
display(summary)